# 전력 예측 결과 · 오차 데이터셋 생성

`LSTM_AIModel.py`의 다변량 LSTM(시나리오 D, 80:20 분할)을 학습한 뒤 **1~9월 전체 구간**을 예측하고,
실제값 · 예측값 · 오차 · 입력 조건을 한 표로 저장한다. 이 결과는 `Error_Driver_Pipeline.ipynb`의 오차 원인 분석 입력으로 쓰인다.

| 섹션 | 출력 파일 | 내용 |
|---|---|---|
| 1 | `full_df_power_only.pkl` | 전력 실제/예측 · 오차 · 오차율 · 과대/과소 · 학습/테스트 구분 |
| 2 | `full_df_power_only.xlsx` | 1과 같은 내용 (엑셀) |
| 3 | `weather_prediction_5cols.xlsx` | 기상 4개 변수(기온·습도·강수량·풍속) LSTM 예측 결과 |

- **입력**: `okm_augumented_2021_processed.xlsx`, `LSTM_AIModel.py`
- 입력 변수: `LSTM_AIModel.FEATURES` + `day`, `is_holiday`, `is_weekend`

## 1. 전력 예측 결과 pkl 생성

| 컬럼 | 의미 |
|---|---|
| `power_true` / `power_pred` | 실제 / 예측 전력(kW) |
| `power_error_signed` | 실제 − 예측 (+ = 과소예측) |
| `power_pct_error` | \|오차\| / 실제 × 100 (실제 ≤ 1kW 이면 NaN) |
| `over_under_pred` | 1 = 과대예측, 0 = 과소예측(또는 동일) |
| `split` | `train(in-sample)` / `test(예측)` |

In [1]:
import numpy as np
import pandas as pd
import warnings
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler

# LSTM_AIModel.py 의 공통 함수 · 상수
from LSTM_AIModel import (
    load_data,
    make_sequences,
    train_lstm_model,
    evaluate_model,
    FEATURES,
    TARGET,
    WINDOW_SIZE
)

# 1. 데이터 불러오기 (1~9월)
df_1_9 = load_data("okm_augumented_2021_processed.xlsx")

# 2. 날짜 파생 변수 (day: 1~7 = 월~일, 8 = 공휴일)
df_1_9["is_holiday"] = (df_1_9["day"] == 8).astype(int)
df_1_9["is_weekend"] = (df_1_9["day"].isin([6, 7])).astype(int)

dow_map = {
    1: "월", 2: "화", 3: "수", 4: "목", 
    5: "금", 6: "토", 7: "일", 8: "공휴일"
}
df_1_9["day_name"] = df_1_9["day"].map(dow_map)

# 3. 입력 변수 = 공통 FEATURES + 날짜 파생 변수
extended_features = FEATURES + ["day", "is_holiday", "is_weekend"]
target = TARGET
target_idx = len(extended_features)  # target을 맨 뒤에 붙이므로 마지막 열

data_values = df_1_9[extended_features + [target]].values

# 4. 학습 / 테스트 분할 (시나리오 D: 80:20)
train_size = int(len(data_values) * 0.8)
train_data = data_values[:train_size]
test_data = data_values[train_size - WINDOW_SIZE:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# 시퀀스 생성
X_train, y_train = make_sequences(train_scaled, target_idx, WINDOW_SIZE)
X_test, y_test = make_sequences(test_scaled, target_idx, WINDOW_SIZE)

# 5. LSTM 학습
model = train_lstm_model(X_train, y_train, (X_train.shape[1], X_train.shape[2]))

# 6. 1~9월 전체 구간 예측 (학습 구간은 in-sample 예측)
full_scaled = scaler.transform(data_values)
X_full, y_full = make_sequences(full_scaled, target_idx, WINDOW_SIZE)

_, y_true_full, y_pred_full = evaluate_model(
    model, X_full, y_full, scaler, target_idx, extended_features
)

# 7. 결과 표 구성 (첫 WINDOW_SIZE 시간은 예측값이 없어 제외)
cond_cols = [
    "datetime", "m", "생산량", "기온", "풍속", "습도", "강수량",
    "전기요금(계절)", "공장인원", "시간", "day", "day_name", "is_holiday", "is_weekend",
    "전력 구간합계", "전력 최대값", "전력 최소값", "전력 변동폭"
]

full_df = df_1_9.iloc[WINDOW_SIZE:].reset_index(drop=True)[cond_cols].copy()
full_df["power_true"] = y_true_full
full_df["power_pred"] = y_pred_full

# 8. 오차 및 오차율 계산
full_df["power_error_signed"] = full_df["power_true"] - full_df["power_pred"]
eps_mask = full_df["power_true"].abs() > 1
full_df["power_pct_error"] = np.where(
    eps_mask,
    (full_df["power_error_signed"].abs() / full_df["power_true"].abs()) * 100,
    np.nan
)

# 과대예측(1) / 과소예측(0) 구분
full_df["over_under_pred"] = np.where(
    full_df["power_true"] < full_df["power_pred"], 
    1, 
    0
)

# 9. 학습 / 테스트 구간 표시
split_point = train_size - WINDOW_SIZE
full_df["split"] = np.where(
    np.arange(len(full_df)) < split_point,
    "train(in-sample)",
    "test(예측)"
)

# 10. pkl 저장
output_filename = "full_df_power_only.pkl"
full_df.to_pickle(output_filename)

/home/kampuser/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/kampuser/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

I0000 00:00:1790680480.589831   44788 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790680480.661817   44788 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790680480.662291   44788 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790680480.666052   44788 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

## 2. 같은 결과를 엑셀로 저장
1과 같은 과정을 거쳐 `full_df_power_only.xlsx`로 저장한다.

In [1]:
import numpy as np
import pandas as pd
import warnings
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler

# LSTM_AIModel.py 의 공통 함수 · 상수
from LSTM_AIModel import (
    load_data,
    make_sequences,
    train_lstm_model,
    evaluate_model,
    FEATURES,
    TARGET,
    WINDOW_SIZE
)

# 1. 데이터 불러오기 (1~9월)
df_1_9 = load_data("okm_augumented_2021_processed.xlsx")

# 2. 날짜 파생 변수 (day: 1~7 = 월~일, 8 = 공휴일)
df_1_9["is_holiday"] = (df_1_9["day"] == 8).astype(int)
df_1_9["is_weekend"] = (df_1_9["day"].isin([6, 7])).astype(int)

dow_map = {
    1: "월", 2: "화", 3: "수", 4: "목", 
    5: "금", 6: "토", 7: "일", 8: "공휴일"
}
df_1_9["day_name"] = df_1_9["day"].map(dow_map)

# 3. 입력 변수 = 공통 FEATURES + 날짜 파생 변수
extended_features = FEATURES + ["day", "is_holiday", "is_weekend"]
target = TARGET
target_idx = len(extended_features)  # target을 맨 뒤에 붙이므로 마지막 열

data_values = df_1_9[extended_features + [target]].values

# 4. 학습 / 테스트 분할 (시나리오 D: 80:20)
train_size = int(len(data_values) * 0.8)
train_data = data_values[:train_size]
test_data = data_values[train_size - WINDOW_SIZE:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# 시퀀스 생성
X_train, y_train = make_sequences(train_scaled, target_idx, WINDOW_SIZE)
X_test, y_test = make_sequences(test_scaled, target_idx, WINDOW_SIZE)

# 5. LSTM 학습
model = train_lstm_model(X_train, y_train, (X_train.shape[1], X_train.shape[2]))

# 6. 1~9월 전체 구간 예측 (학습 구간은 in-sample 예측)
full_scaled = scaler.transform(data_values)
X_full, y_full = make_sequences(full_scaled, target_idx, WINDOW_SIZE)

_, y_true_full, y_pred_full = evaluate_model(
    model, X_full, y_full, scaler, target_idx, extended_features
)

# 7. 결과 표 구성 (첫 WINDOW_SIZE 시간은 예측값이 없어 제외)
cond_cols = [
    "datetime", "m", "생산량", "기온", "풍속", "습도", "강수량",
    "전기요금(계절)", "공장인원", "시간", "day", "day_name", "is_holiday", "is_weekend",
    "전력 구간합계", "전력 최대값", "전력 최소값", "전력 변동폭"
]

full_df = df_1_9.iloc[WINDOW_SIZE:].reset_index(drop=True)[cond_cols].copy()
full_df["power_true"] = y_true_full
full_df["power_pred"] = y_pred_full

# 8. 오차 및 오차율 계산
full_df["power_error_signed"] = full_df["power_true"] - full_df["power_pred"]
eps_mask = full_df["power_true"].abs() > 1
full_df["power_pct_error"] = np.where(
    eps_mask,
    (full_df["power_error_signed"].abs() / full_df["power_true"].abs()) * 100,
    np.nan
)

# 과대예측(1) / 과소예측(0) 구분
full_df["over_under_pred"] = np.where(
    full_df["power_true"] < full_df["power_pred"], 
    1, 
    0
)

# 9. 학습 / 테스트 구간 표시
split_point = train_size - WINDOW_SIZE
full_df["split"] = np.where(
    np.arange(len(full_df)) < split_point,
    "train(in-sample)",
    "test(예측)"
)

# 10. 엑셀 저장
output_filename = "full_df_power_only.xlsx"
full_df.to_excel(output_filename, index=False, engine="openpyxl")

print(f"추론 결과 저장 완료: {output_filename}")

/home/kampuser/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/kampuser/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

I0000 00:00:1790922481.325576     184 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790922481.383857     184 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790922481.384216     184 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790922481.387052     184 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

추론 결과 저장 완료: full_df_power_only.xlsx


## 3. 기상 데이터 예측 (다중 출력 LSTM)
생산·전력·기상 변수의 직전 24시간으로 다음 1시간의 **기온 · 습도 · 강수량 · 풍속 4개를 동시에** 예측한다.
- 분할: 학습 80% / 테스트 20%
- 출력: `weather_prediction_5cols.xlsx` (timestamp + 변수별 실제 · 예측 · 오차)

In [4]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 불러오기
input_file = "okm_augumented_2021_processed.xlsx"
df = pd.read_excel(input_file)

df["datetime"] = pd.to_datetime(df["datetime"])

# 날짜 파생 변수
df["is_holiday"] = (df["day"] == 8).astype(int)
df["is_weekend"] = (df["day"].isin([6, 7])).astype(int)

# 2. 입력 변수 / 예측 대상(4개)
TARGET_COLS = ["기온", "습도", "강수량", "풍속"]

# 입력에 과거 기상 4개 변수도 포함해 서로의 영향을 함께 학습
FEATURE_COLS = [
    "생산량", "공장인원", "시간", "day", "is_holiday", "is_weekend",
    "전력 구간합계", "전력 최대값", "전력 최소값", "전력 변동폭"
] + TARGET_COLS

WINDOW_SIZE = 24  # 직전 24시간 입력

# 3. 스케일링 (입력 / 출력 별도 스케일러)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaled_X = scaler_X.fit_transform(df[FEATURE_COLS])
scaled_y = scaler_y.fit_transform(df[TARGET_COLS])

# 4. 슬라이딩 윈도우 시퀀스 생성
def make_multivariate_sequences(X_data, y_data, window_size):
    X_seq, y_seq = [], []
    for i in range(len(X_data) - window_size):
        X_seq.append(X_data[i : i + window_size])
        y_seq.append(y_data[i + window_size])
    return np.array(X_seq), np.array(y_seq)

# 5. 학습 80% / 테스트 20% 분할
train_size = int(len(df) * 0.8)

train_X_raw = scaled_X[:train_size]
train_y_raw = scaled_y[:train_size]
test_X_raw = scaled_X[train_size - WINDOW_SIZE:]
test_y_raw = scaled_y[train_size - WINDOW_SIZE:]

X_train, y_train = make_multivariate_sequences(train_X_raw, train_y_raw, WINDOW_SIZE)
X_test, y_test = make_multivariate_sequences(test_X_raw, test_y_raw, WINDOW_SIZE)

# 6. LSTM 모델 구축 및 학습
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(len(TARGET_COLS))  # 기온 · 습도 · 강수량 · 풍속 동시 출력
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
print("LSTM 모델 학습 중...")
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    verbose=1
)

# 7. 전체 구간 예측
X_full, y_full_scaled = make_multivariate_sequences(scaled_X, scaled_y, WINDOW_SIZE)
y_pred_scaled = model.predict(X_full)

# 원래 단위로 역변환
y_true_full = scaler_y.inverse_transform(y_full_scaled)
y_pred_full = scaler_y.inverse_transform(y_pred_scaled)

# 8. 결과 표: timestamp + 기상 변수별 실제 · 예측 · 오차
full_df = pd.DataFrame()
full_df["timestamp"] = df.iloc[WINDOW_SIZE:]["datetime"].reset_index(drop=True)

for i, col in enumerate(TARGET_COLS):
    full_df[f"{col}_true"] = y_true_full[:, i]
    full_df[f"{col}_pred"] = y_pred_full[:, i]
    full_df[f"{col}_error"] = full_df[f"{col}_true"] - full_df[f"{col}_pred"]

# 학습 / 테스트 구간 표시
split_point = train_size - WINDOW_SIZE
full_df["split"] = np.where(
    np.arange(len(full_df)) < split_point,
    "train",
    "test"
)

# 9. 엑셀 저장
output_filename = "weather_prediction_5cols.xlsx"
full_df.to_excel(output_filename, index=False, engine="openpyxl")

print(f"\n추론 결과 저장 완료: {output_filename}")

LSTM 모델 학습 중...
Epoch 1/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0635 - mae: 0.1771 - val_loss: 0.0135 - val_mae: 0.0750
Epoch 2/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0185 - mae: 0.1041 - val_loss: 0.0100 - val_mae: 0.0608
Epoch 3/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0142 - mae: 0.0896 - val_loss: 0.0093 - val_mae: 0.0584
Epoch 4/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0120 - mae: 0.0818 - val_loss: 0.0083 - val_mae: 0.0546
Epoch 5/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0107 - mae: 0.0767 - val_loss: 0.0079 - val_mae: 0.0554
Epoch 6/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0099 - mae: 0.0734 - val_loss: 0.0075 - val_mae: 0.0556
Epoch 7/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0090 - mae: 0.0696 - val_loss: 0.0067 - val_mae: 0.0527
Epoch 8/30
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0083 - mae: 0.0667 - val_loss: 0.0062 - val_mae: 0.0491
Epoch 9/30
154/154 ━━━━━━━━━━━━━━